# Out-of-sample performance of industry portfolio strategies

We compare eight ways to allocate wealth across ten US industry portfolios. Each month, we estimate portfolio weights using the preceding 240 months of data and hold those weights for one month. All portfolios invest 100% of wealth in the risky assets. Some strategies allow short positions.

The question is whether estimating expected returns, covariances, or factor exposures improves the investor's realized certainty equivalent relative to equal weighting. We evaluate performance both before and after transaction costs of 50 basis points per dollar traded.


In [10]:
from io import StringIO
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from IPython.display import display, Markdown

# Experiment parameters (assignment: January 1965 - January 2025, M = 240)
START = "1965-01"
END = "2025-01"
WINDOW = 240    # estimation window in months
COST = 0.005    # proportional transaction cost of 50 basis points
THETA = 2       # timing parameter of the reward-to-risk timing portfolio

# Set up project and data directories
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

INDUSTRY_FILE = DATA_DIR / "10_Industry_Portfolios.csv"
FF3_FILE = DATA_DIR / "F-F_Research_Data_Factors.csv"
MOM_FILE = DATA_DIR / "F-F_Momentum_Factor.csv"

for file in [INDUSTRY_FILE, FF3_FILE, MOM_FILE]:
    assert file.exists(), f"Missing input file: {file}"


## 1. Data and return definitions

We use monthly returns on ten US industry portfolios (value-weighted), the US Fama–French three factors (Mkt-RF, SMB, HML), the US momentum factor (Mom), and the one-month risk-free rate. All series come from Kenneth French's Data Library. The sample runs from January 1965 to January 2025 (721 months). The library reports returns in percent; all calculations below use decimal returns.


 Let $r^g_{i,t}$ denote the return of industry $i$ and $r_{f,t}$ the risk-free rate. The excess return is

$$
r_{i,t}=r^g_{i,t}-r_{f,t},\qquad r_t=(r_{1,t},\dots,r_{N,t})',\qquad N=10.
$$

We estimate the portfolio weights from the excess returns $r_t$. Every portfolio is fully invested ($\iota'w=1$), so its return is $r^g_{p,t}=w'r_t+r_{f,t}$ (Lecture 2). The risk-free rate shifts all portfolio returns by the same amount and does not change the optimal weights. We keep the returns $r^g_{i,t}$ to track how the value of each holding drifts between rebalances, which we need for turnover.

The factors $f_t=(\text{Mkt-RF},\text{SMB},\text{HML},\text{Mom})'$ are excess or long-short returns and need no adjustment. They enter only the reward-to-risk timing portfolio in part (c).

In [2]:
def read_industry_file(path):
    """Read the industry CSV: one row per month, 'yyyymm' index, returns in percent."""
    data = pd.read_csv(path, index_col="yyyymm")
    data.index = pd.to_datetime(data.index.astype(str), format="%Y%m").to_period("M")
    data.columns = data.columns.str.strip()
    return data


def read_french_monthly(path, columns):
    """Read the monthly block of a Kenneth French CSV file (returns in percent).

    The raw file has a text header, a monthly block (rows start with YYYYMM)
    and an annual block (rows start with YYYY). We keep the monthly rows only.
    """
    lines = path.read_text(encoding="utf-8-sig", errors="replace").splitlines()
    is_monthly = [bool(re.match(r"^\s*\d{6}\s*,", line)) for line in lines]
    first = is_monthly.index(True)

    # The last non-empty line above the first monthly row holds the column names.
    header_line = next(line for line in reversed(lines[:first]) if line.strip())
    header = [name.strip() for name in header_line.split(",")[1:]]
    assert [h.lower() for h in header] == [c.lower() for c in columns], (
        f"{path.name}: expected columns {columns}, found {header}"
    )

    monthly = "\n".join(line for line, keep in zip(lines, is_monthly) if keep)
    data = pd.read_csv(StringIO(monthly), header=None)
    data.columns = ["date", *columns]
    data.index = pd.to_datetime(
        data.pop("date").astype(str), format="%Y%m"
    ).dt.to_period("M")
    return data


industry_pct = read_industry_file(INDUSTRY_FILE)
ff3_pct = read_french_monthly(FF3_FILE, ["Mkt-RF", "SMB", "HML", "RF"])
mom_pct = read_french_monthly(MOM_FILE, ["Mom"])

months = pd.period_range(START, END, freq="M")
industries = industry_pct.columns.tolist()
factor_names = ["Mkt-RF", "SMB", "HML", "Mom"]

# Reindex to the required calendar so that a missing month becomes visible.
for source in [industry_pct, ff3_pct, mom_pct]:
    assert source.index.is_unique, "Duplicate month in an input file."

data_pct = pd.concat([industry_pct, ff3_pct, mom_pct], axis=1).reindex(months)
data_pct = data_pct.replace([-99.99, -999, -999.99], np.nan)
assert data_pct.notna().all().all(), "Missing return or factor in the sample."
assert len(industries) == 10 and len(data_pct) == 721

data = data_pct / 100                                   # percent -> decimal
total_returns = data[industries]                        # r^g_t
risk_free = data["RF"]                                  # r_f,t
excess_returns = total_returns.sub(risk_free, axis="index")   # r_t = r^g_t - r_f,t
factors = data[factor_names]                            # f_t

print(f"Sample: {months[0]} to {months[-1]} ({len(months)} months)")
print(f"First estimation window: {months[0]} to {months[WINDOW - 1]}")
print(f"Holding period: {months[WINDOW]} to {months[-1]}")
print(f"Out-of-sample months: {len(months) - WINDOW}")


Sample: 1965-01 to 2025-01 (721 months)
First estimation window: 1965-01 to 1984-12
Holding period: 1985-01 to 2025-01
Out-of-sample months: 481


In [3]:
# Units and alignment checks.
assert risk_free.between(0, 0.02).all(), "RF is not a monthly decimal rate."

# Market factor vs. equal-weighted average industry excess return: both measure the
# US equity market, so a shifted month in either file would destroy this correlation.
corr = factors["Mkt-RF"].corr(excess_returns.mean(axis=1))
assert corr > 0.9, f"Mkt-RF and industry returns look misaligned (corr = {corr:.2f})."
print(f"Correlation of Mkt-RF with the equal-weighted industry excess return: {corr:.3f}")

# Descriptive statistics, annualized, in percent.
panel = pd.concat([excess_returns, factors], axis=1)
stats = pd.DataFrame({
    "Mean (% p.a.)": 1200 * panel.mean(),
    "Volatility (% p.a.)": 100 * np.sqrt(12) * panel.std(),
})
stats["Sharpe ratio"] = stats["Mean (% p.a.)"] / stats["Volatility (% p.a.)"]
display(stats.style.format("{:.2f}"))


Correlation of Mkt-RF with the equal-weighted industry excess return: 0.983


,Mean (% p.a.),Volatility (% p.a.),Sharpe ratio
NoDur,7.65,14.78,0.52
Durbl,7.88,25.61,0.31
Manuf,7.13,17.42,0.41
Enrgy,7.60,21.19,0.36
HiTec,8.85,22.19,0.40
Telcm,5.32,16.43,0.32
Shops,8.39,18.05,0.46
Hlth,8.09,16.72,0.48
Utils,5.58,14.23,0.39
Other,7.20,18.59,0.39


## 2. Portfolio weights

At the start of each holding month, let $\hat\mu$ and $\hat\Sigma$ be the sample mean and sample covariance matrix of the ten industry excess returns over the preceding $M=240$ months, and let $\iota$ be the $N\times 1$ vector of ones. All strategies are fully invested, $\iota'w=1$. Two scalars appear repeatedly, we conveniently rename them:

$$
d=\iota'\hat\Sigma^{-1}\iota,\qquad a=\iota'\hat\Sigma^{-1}\hat\mu .
$$

### (a) Equal weight, global minimum variance, and tangency

$$
w_{ew}=\frac{\iota}{N},\qquad
w_{gmv}=\frac{\hat\Sigma^{-1}\iota}{d},\qquad
w_{tan}=\frac{\hat\Sigma^{-1}\hat\mu}{a}.
$$

Equal weighting estimates no parameters. GMV uses the covariance matrix only. The tangency portfolio also uses the expected excess returns, which are much harder to estimate. The scalar $a$ equals $d$ times the sample mean excess return of the GMV portfolio, so it is a noisy number. When $a$ is close to zero, normalizing the tangency weights to sum to one produces large offsetting long and short positions. When $a$ is negative, the normalization reverses the sign of all weights.

### (b) Long-only GMV and certainty-equivalent portfolios

The long-only GMV portfolio solves

$$
\min_w\; w'\hat\Sigma w
\quad\text{subject to}\quad
\iota'w=1,\quad w_i\geq 0 .
$$

It has no closed-form solution, hence we solve it numerically.


The portfolio mixes the tangency and GMV portfolios. A higher risk aversion lowers $\kappa_\gamma$ and moves the portfolio toward GMV. 

In [4]:
def long_only_gmv(cov):
    """Minimum-variance weights subject to full investment and no short sales."""
    n = len(cov)
    equal_weights = np.full(n, 1 / n)
    scaled_cov = cov / np.diag(cov).mean()    # rescale for numerical stability

    result = minimize(
        fun=lambda w: w @ scaled_cov @ w,
        x0=equal_weights,
        jac=lambda w: 2 * scaled_cov @ w,
        method="SLSQP",
        bounds=[(0, 1)] * n,
        constraints={
            "type": "eq",
            "fun": lambda w: w.sum() - 1,
            "jac": lambda w: np.ones(n),
        },
        options={"ftol": 1e-12, "maxiter": 500},
    )

    if not result.success:
        raise RuntimeError(f"Long-only GMV failed: {result.message}")

    # Remove numerical noise: exact zeros instead of -1e-17, weights sum to one.
    w = np.clip(result.x, 0, None)
    return w / w.sum()


def markowitz_weights(mu, cov):
    """Compute the six portfolio rules of parts (a) and (b)."""
    n = len(mu)
    iota = np.ones(n)

    # solve(S, v) returns S^{-1} v without forming the inverse explicitly.
    sigma_inv_iota = np.linalg.solve(cov, iota)
    sigma_inv_mu = np.linalg.solve(cov, mu)

    d = sigma_inv_iota.sum()      # iota' Sigma^{-1} iota
    a = sigma_inv_mu.sum()        # iota' Sigma^{-1} mu
    if np.isclose(a, 0, atol=1e-12, rtol=0):
        raise ValueError("Tangency normalization is numerically undefined.")

    gmv = sigma_inv_iota / d
    tan = sigma_inv_mu / a

    def ce(gamma):
        return sigma_inv_mu / gamma + (1 - a / gamma) * gmv

    return {
        "EW": iota / n,
        "GMV": gmv,
        "GMV+": long_only_gmv(cov),
        "TAN": tan,
        "CE(4)": ce(4),
        "CE(8)": ce(8),
    }


### (c) Reward-to-risk timing based on average factor exposure

Reward-to-risk timing (Kirby and Ostdiek, 2012) sets weights proportional to $(\mu_i/\sigma_i^2)^\vartheta$. We replace the expected excess return $\mu_i$ by the industry's average factor exposure. Within each estimation window, we regress each industry's excess return on the same four factors:

$$
r_{i,s}=\alpha_i+\beta_{i,M}(\text{Mkt-RF})_s+\beta_{i,S}\,\text{SMB}_s
+\beta_{i,H}\,\text{HML}_s+\beta_{i,U}\,\text{Mom}_s+\varepsilon_{i,s}.
$$

For each industry, $\bar\beta_i$ is the arithmetic average of the four estimated slopes. With $\hat\sigma_i^2=\hat\Sigma_{ii}$ and timing parameter $\vartheta=2$,

$$
w_{rrt,i}=\frac{q_i}{\sum_j q_j},
\qquad
q_i=\left(\frac{\max(\bar\beta_i,0)}{\hat\sigma_i^2}\right)^{\vartheta}.
$$

We take the positive part before applying the power, since a negative $\bar\beta_i$ would be squared into a positive score and receive a positive weight. The rule is long-only by construction and needs no matrix inversion. The denominator is the industry's total excess-return variance, which includes both factor and residual risk.
### (d) Own strategy: inverse-variance weighting

We choose a long-only portfolio that gives more weight to industries with lower estimated variance:

$$
w_{own,i}=
\frac{1/\hat\sigma_i^2}{\sum_j 1/\hat\sigma_j^2}.
$$

This is the GMV solution when the covariance matrix is approximated by its diagonal. It provides a direct comparison with unrestricted GMV. With ten industries, the rule estimates ten variances and omits 45 pairwise covariances. This can reduce sensitivity to estimation error, at the cost of ignoring diversification opportunities between industries. The rule is fixed before inspecting out-of-sample performance.

In [5]:
def timing_weights(window_excess, window_factors, variances, theta=THETA):
    """Compute the average-beta RRT portfolio and the inverse-variance portfolio."""
    # One intercept plus four factor columns.
    X = np.column_stack([np.ones(len(window_factors)), window_factors])

    # Each column of Y is one industry: this fits ten separate OLS regressions.
    Y = window_excess
    coefficients, _, rank, _ = np.linalg.lstsq(X, Y, rcond=None)
    if rank != X.shape[1]:
        raise ValueError("Factor regression has a rank-deficient design matrix.")

    slopes = coefficients[1:, :]          # 4 factors x 10 industries
    average_beta = slopes.mean(axis=0)    # one average per industry

    positive_beta = np.maximum(average_beta, 0)
    rrt_score = (positive_beta / variances) ** theta
    if rrt_score.sum() == 0:
        raise ValueError("All RRT scores are zero in this window.")

    inverse_variance = 1 / variances

    return {
        "RRT": rrt_score / rrt_score.sum(),
        "OWN": inverse_variance / inverse_variance.sum(),
    }


## 3. Rolling out-of-sample experiment

For holding month $t$, we estimate all inputs from months $t-240,\ldots,t-1$. We then apply the resulting weights $w_t$ to the realized industry excess returns in month $t$:

$$
x_{p,t}=w_t'x_t.
$$

For example, the January 1985 weights use only January 1965 through December 1984. January 1985 enters the estimation sample for the first time when we choose the February 1985 weights.

We save every target weight. This allows us to reconstruct realized returns, measure rebalancing trades, and check the optimization results independently.

In [6]:
strategy_names = ["EW", "GMV", "GMV+", "TAN", "CE(4)", "CE(8)", "RRT", "OWN"]
holding_months = months[WINDOW:]

weights = {
    name: pd.DataFrame(index=holding_months, columns=industries, dtype=float)
    for name in strategy_names
}

for t in range(WINDOW, len(months)):
    holding_month = months[t]

    # The right endpoint t is excluded from both training slices.
    window_excess = excess_returns.iloc[t - WINDOW:t]
    window_factors = factors.iloc[t - WINDOW:t]

    mu = window_excess.mean().to_numpy()
    cov = window_excess.cov().to_numpy()

    current = markowitz_weights(mu, cov)
    current.update(
        timing_weights(
            window_excess.to_numpy(),
            window_factors.to_numpy(),
            np.diag(cov),
        )
    )

    # In sample, the unrestricted GMV has the largest feasible set.
    # Equal weights are feasible for the long-only GMV problem.
    var_gmv = current["GMV"] @ cov @ current["GMV"]
    var_long = current["GMV+"] @ cov @ current["GMV+"]
    var_ew = current["EW"] @ cov @ current["EW"]

    assert var_gmv <= var_long + 1e-10, holding_month
    assert var_long <= var_ew + 1e-10, holding_month

    for name, w in current.items():
        assert np.isfinite(w).all(), (holding_month, name)
        assert np.isclose(w.sum(), 1, atol=1e-8, rtol=0), (holding_month, name)
        weights[name].loc[holding_month] = w

for name in ["GMV+", "RRT", "OWN"]:
    assert weights[name].to_numpy().min() >= -1e-8, name

np.testing.assert_allclose(
    weights["CE(8)"],
    0.5 * (weights["CE(4)"] + weights["GMV"]),
    atol=1e-10,
)

print(f"Verified {len(holding_months)} monthly allocations for each strategy.")

Verified 481 monthly allocations for each strategy.


## 4. Turnover, transaction costs, and certainty equivalents

An industry's portfolio weight changes during a month as its return differs from the portfolio return. If $w_{i,t}$ is its starting weight, its weight at month-end is

$$
w^{drift}_{i,t}
=\frac{w_{i,t}(1+R_{i,t})}{1+w_t'R_t}.
$$

The turnover required to establish the next target is

$$
TO_{t+1}=\sum_i |w_{i,t+1}-w^{drift}_{i,t}|.
$$

We count both purchases and sales, as in the turnover definition in Lecture 6. With a proportional transaction cost of $c=0.005$, the net excess return for holding month $t$ is

$$
x^{net}_{p,t}=x_{p,t}-c\,TO_t.
$$

We treat the first portfolio as already established. January 1985 therefore incurs no opening cost. Average turnover uses the 480 observable rebalances from February 1985 to January 2025. Gross and net CE both use the same 481 holding months.

For a series of monthly excess returns $z_t$, the annualized certainty equivalent in percentage points is

$$
CE^{ann}(\gamma)
=100\cdot12\left(\bar z-\frac{\gamma}{2}s_z^2\right),
$$

where $s_z^2$ is the sample variance. We apply the formula separately to gross and net returns. Trading costs can change both the mean and the variance.

In [7]:
oos_total = total_returns.loc[holding_months]
oos_excess = excess_returns.loc[holding_months]

gross_excess = pd.DataFrame(index=holding_months, columns=strategy_names, dtype=float)
turnover = gross_excess.copy()

for name, w in weights.items():
    gross_excess[name] = (w * oos_excess).sum(axis=1)
    portfolio_total = (w * oos_total).sum(axis=1)

    # A nonpositive gross wealth factor would invalidate ordinary weight drift.
    if (1 + portfolio_total <= 0).any():
        failed_months = portfolio_total.index[1 + portfolio_total <= 0].tolist()
        raise ValueError(f"{name}: nonpositive portfolio wealth in {failed_months}.")

    drifted = (w * (1 + oos_total)).div(1 + portfolio_total, axis="index")

    # At the start of t, compare target t to drifted holdings from t-1.
    # min_count preserves the missing first rebalance.
    turnover[name] = (w - drifted.shift(1)).abs().sum(
        axis=1, min_count=len(industries)
    )

assert turnover.iloc[0].isna().all()
assert turnover.iloc[1:].notna().all().all()

trading_cost = COST * turnover.copy()
trading_cost.iloc[0] = 0.0
net_excess = gross_excess - trading_cost

# Independent benchmarks calculated from the uploaded industry CSV.
january_1985_ew_total = oos_total.iloc[0].mean()
np.testing.assert_allclose(january_1985_ew_total, 0.08176, atol=1e-10)
np.testing.assert_allclose(turnover["EW"].iloc[1], 0.02415323177, atol=1e-9)
np.testing.assert_allclose(turnover["EW"].mean(), 0.02459239894, atol=1e-9)

print(f"January 1985 EW total return: {january_1985_ew_total:.2%}")
print(f"February 1985 EW turnover:    {turnover['EW'].iloc[1]:.2%}")
print(f"Average monthly EW turnover:  {turnover['EW'].mean():.2%}")

January 1985 EW total return: 8.18%
February 1985 EW turnover:    2.42%
Average monthly EW turnover:  2.46%


## 5. Table 1. Portfolio performance

Each row represents one portfolio strategy. The two gross CE columns evaluate the same realized returns at two levels of investor risk aversion. The same holds for the two net CE columns. In particular, the portfolio constructed using $\gamma=4$ is evaluated at both $\gamma=4$ and $\gamma=8$.

CE values are annual percentages. Turnover is the average percentage of portfolio value traded per month. All reported figures are rounded to two decimals after calculation.

In [8]:
def annual_ce(returns, gamma):
    monthly_mean = returns.mean()
    monthly_variance = returns.var(ddof=1)
    return 100 * 12 * (monthly_mean - 0.5 * gamma * monthly_variance)


table1 = pd.DataFrame({
    "Gross CE, γ=4 (%)": annual_ce(gross_excess, 4),
    "Gross CE, γ=8 (%)": annual_ce(gross_excess, 8),
    "Monthly turnover (%)": 100 * turnover.mean(),
    "Net CE, γ=4 (%)": annual_ce(net_excess, 4),
    "Net CE, γ=8 (%)": annual_ce(net_excess, 8),
}).loc[strategy_names]

table1.index.name = "Portfolio"

# Raising evaluation risk aversion from 4 to 8 has a known exact effect.
np.testing.assert_allclose(
    table1["Gross CE, γ=4 (%)"] - table1["Gross CE, γ=8 (%)"],
    2400 * gross_excess.var(ddof=1),
)
np.testing.assert_allclose(
    table1["Net CE, γ=4 (%)"] - table1["Net CE, γ=8 (%)"],
    2400 * net_excess.var(ddof=1),
)

display(table1.style.format("{:.2f}"))

,"Gross CE, γ=4 (%)","Gross CE, γ=8 (%)",Monthly turnover (%),"Net CE, γ=4 (%)","Net CE, γ=8 (%)"
Portfolio,,,,,
EW,5.13,0.71,2.46,4.98,0.56
GMV,5.90,2.76,9.02,5.36,2.22
GMV+,5.57,2.44,3.27,5.37,2.24
TAN,1.11,-7.05,50.18,-1.85,-9.97
CE(4),-2.25,-12.27,62.53,-6.00,-16.03
CE(8),3.54,-1.32,29.10,1.80,-3.06
RRT,4.99,0.87,2.89,4.81,0.70
OWN,5.39,1.58,2.34,5.25,1.44


In [9]:
net4 = table1["Net CE, γ=4 (%)"]
net8 = table1["Net CE, γ=8 (%)"]
monthly_to = table1["Monthly turnover (%)"]

best4 = net4.idxmax()
best8 = net8.idxmax()

gmv_ce_change = net8["GMV+"] - net8["GMV"]
gmv_to_change = monthly_to["GMV+"] - monthly_to["GMV"]
own_ce_gap = net8["OWN"] - net8["EW"]

def direction(value, positive, negative):
    if np.isclose(value, 0, atol=1e-10):
        return "was unchanged"
    return f"{positive if value > 0 else negative} by {abs(value):.2f} percentage points"


summary = (
    f"At risk aversion 4, {best4} achieved the highest annual net CE "
    f"({net4[best4]:.2f}%, versus {net4['EW']:.2f}% for equal weighting). "
    f"At risk aversion 8, {best8} ranked first at {net8[best8]:.2f}%. "
    f"After imposing the long-only constraint on GMV, monthly turnover "
    f"{direction(gmv_to_change, 'rose', 'fell')} and annual net CE at risk aversion 8 "
    f"{direction(gmv_ce_change, 'rose', 'fell')}. "
    f"Relative to equal weighting, annual net CE for inverse-variance weighting "
    f"{direction(own_ce_gap, 'was higher', 'was lower')} at risk aversion 8."
)

display(Markdown("### (e) Key findings\n\n" + summary))
print("Summary word count:", len(summary.split()))

tan_exposure = weights["TAN"].abs().sum(axis=1)
print(
    f"Tangency diagnostic: maximum gross exposure was "
    f"{tan_exposure.max():.2f} times wealth in {tan_exposure.idxmax()}."
)

### (e) Key findings

At risk aversion 4, GMV+ achieved the highest annual net CE (5.37%, versus 4.98% for equal weighting). At risk aversion 8, GMV+ ranked first at 2.24%. After imposing the long-only constraint on GMV, monthly turnover fell by 5.75 percentage points and annual net CE at risk aversion 8 rose by 0.02 percentage points. Relative to equal weighting, annual net CE for inverse-variance weighting was higher by 0.89 percentage points at risk aversion 8.

Summary word count: 73
Tangency diagnostic: maximum gross exposure was 17.57 times wealth in 1985-02.
